# Regression models — our day-ahead residual-load forecast vs. SMARD

Implements [`.claude/specs/05-regression-models.md`](../../.claude/specs/05-regression-models.md).

We forecast `residual_load` for every hour of a delivery day `DAY`, **issued at 18:00 on `DAY−1`**: the
same point in time as SMARD's public day-ahead forecast. Every model is scored against SMARD's
`fc_residual_load` and against a seasonal-naive floor, on **identical hours**.

**Our models are post-processors of SMARD's forecast.** They take SMARD's published component
forecasts (`fc_grid_load`, `fc_gen_wind_solar`) as inputs. If one wins, the honest claim is "we
reduce SMARD's error by X %", not "we forecast better than the TSOs".

## How to use this notebook

- Change a value in the configuration cells (§1.3), then re-run top to bottom. No later cell
  hardcodes a value the configuration holds.
- The interpretation is written **once**, in the closing section, for the default configuration.
  Tables and plots in between carry titles and units, no commentary.

## What this notebook produces

- day-ahead forecasts from seasonal naive `DAY−7`, `sarimax_fourier` and LightGBM direct / hybrid
  (XGBoost direct / hybrid when switched on), each under a **static** and a **rolling** split method
  on the same test year
- an empirical 95 % prediction interval per model, with its measured coverage
- one scoreboard (accuracy, extremes, intervals) including SMARD and seasonal naive
- two optional exports in `data/models/`, written only when `EXPORT_ENABLED` is on (default off)

## Conventions

- **Sign:** `error = forecast − actual`, as in spec 04. **Positive = over-forecast.**
- **Units:** hourly readings, forecasts and errors in `MWh`; capacity in `MW`; skill and coverage in
  `%`. The `MW` relabelling in `team-EDA.ipynb` does not apply here.
- **Durations, never row counts.** No literal calendar year or date appears in code.
- `time_series` holds exactly `SERIES + DERIVED`. Everything this notebook builds lives in separate
  frames.

## Not in this notebook

- Holt-Winters, seasonal ARIMA with a period `m`, MAPE, weather data, `fc_residual_load` as a feature
- risk flags on our forecast (parked [04.3](../../.claude/specs/04.3-risk-label-link.md)) and the
  remaining baselines of parked [04.1](../../.claude/specs/04.1-naive-baseline.md)
- significance tests, MLflow, any change to `modeling/`, reBAP

---

## 1 Setup and configuration

### 1.1 Shared setup (inherited)

Same setup as [`team-EDA.ipynb`](../01_eda/team-EDA.ipynb) §1, as reused by
[`risk-definition.ipynb`](../03_risk_classification/risk-definition.ipynb) and
[`forecast-metrics-claude.ipynb`](../02_forecast_metrics/forecast-metrics-claude.ipynb). Inherited,
not re-derived:

- the data-directory resolver (walks **upward** from the working directory)
- loading, renaming, the German-CSV float conversion and the dtype asserts
- `time_series`, `SERIES`, `DERIVED`, `YEARS`, `DAY_NAMES`, the season mapping
- `style_timeseries` (`ylabel` required)
- the duration pattern and the day-completeness rule from `risk-definition.ipynb` §3.1

Not needed here, so not repeated: `_complete_periods`, `period_mean`, `period_energy`,
`seasonal_plot` and the team-EDA colour configuration. Model colours live in the registry (§1.3).

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it with
`notebooks/API-connection.ipynb`.

In [ ]:
import itertools
import time
import warnings
from pathlib import Path

import holidays
import lightgbm as lgb
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
import xgboost as xgb
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import kpss

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(
    f"pandas {pd.__version__} · numpy {np.__version__} · statsmodels {statsmodels.__version__} · "
    f"lightgbm {lgb.__version__} · xgboost {xgb.__version__}"
)
print(f"Data directory: {DATA}")

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
    "Capacity Wind Offshore": "cap_wind_off",
    "Capacity Wind Onshore": "cap_wind_on",
    "Capacity Solar": "cap_solar"
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot taken before any other cell can touch the frame, so the closing self-check can prove
# nothing in between mutated it.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
    "cap_wind_off", "cap_wind_on", "cap_solar"
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.2 Resolution and durations

Every window, lag, refit interval, Fourier period, issue time and publication lag in this notebook is
a **duration**, converted to an observation count from the **measured** resolution by `n_obs`. A
switch to SMARD's quarter-hour data would change the counts, not the definitions.

At a sub-hourly resolution, models would be compared with each other at native resolution, and with
SMARD only after aggregating to hourly means, because `smard_forecast_errors_hourly.csv` holds
hourly errors. This notebook runs on the hourly `data/smard.csv` only.

In [ ]:
# Resolution is measured, not assumed.
RESOLUTION = time_series.index.to_series().diff().mode().iloc[0]

DAY_COMPLETENESS = 23 / 24   # accepts the spring-DST day, rejects materially short days


def n_obs(duration):
    """Observation count of `duration` at the measured resolution. Refuses a non-multiple."""
    count = duration / RESOLUTION
    if count != int(count):
        raise ValueError(f"{duration} is not a whole multiple of the resolution {RESOLUTION}")
    return int(count)


def describe(value):
    """Short readable form of a configuration value: hours below two days, whole days above."""
    if isinstance(value, pd.Timedelta):
        if value >= pd.Timedelta(days=2) and value == pd.Timedelta(days=value.days):
            return f"{value.days} days"
        return f"{value / pd.Timedelta(hours=1):g} h"
    if isinstance(value, pd.DateOffset):
        return ", ".join(f"{n} {unit.rstrip('s') if n == 1 else unit}" for unit, n in value.kwds.items())
    return str(value)


EXPECTED_OBS_PER_DAY = n_obs(pd.Timedelta(days=1))
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

print(f"resolution        : {describe(RESOLUTION)}  ->  {EXPECTED_OBS_PER_DAY} observations per full day")
print(f"day completeness  : >= {MIN_OBS_PER_DAY} of {EXPECTED_OBS_PER_DAY} observations")

### 1.3 Configuration

Every value a later cell depends on is set here and nowhere else. Edit these the way you edit a
colour map, then re-run the notebook top to bottom. The defaults are the spec's.

| Cell | Holds |
|---|---|
| `DATA_INFO` | the forecast setting: issue time on `DAY−1`, the actuals publication lag, the capacity publication rule |
| `WINDOWS`, `TRAIN_TEST_SPLIT_METHOD` | test, validation and training window lengths, the refit interval, which split methods run |
| `MODELS` | the model registry: per model an on/off switch, family, architecture, fixed parameters, tuning grid, label and colour |
| `FEATURES` | one switch per booster feature group (**provisional**) and the durations the groups use |
| `INTERVAL`, `PLOT_MODEL`, `PLOT_SPLIT_METHOD`, `EXPORT_ENABLED` | interval level, forecast-plot selection, export toggle |

Windows and times are durations: `pd.Timedelta`, or `pd.DateOffset` where calendar months are meant.

In [ ]:
DATA_INFO = {
    "issue_days_before": pd.Timedelta(days=1),     # the forecast for DAY is issued on DAY−1 ...
    "issue_clock": pd.Timedelta(hours=18),         # ... at 18:00 local, once both SMARD components are out
    "actuals_lag": pd.Timedelta(hours=3),          # smard.de shows actuals ~2 h behind real time, +1 h margin
    "capacity_published_after": pd.Timedelta(0),   # year Y's cap_* value: 1 January of Y, 00:00
}

WINDOWS = {
    "test": pd.Timedelta(days=365),         # the last 365 complete delivery days
    "validation": pd.Timedelta(days=365),   # the 365 delivery days before the test window
    "train": pd.DateOffset(months=24),      # calendar months, not 730 days
    "refit_every": pd.Timedelta(days=30),   # rolling method only
}

TRAIN_TEST_SPLIT_METHOD = {"static": True, "rolling": True}

# Clock times implied by DATA_INFO, printed so the rule can be read without doing the arithmetic.
cutoff_clock = DATA_INFO["issue_clock"] - DATA_INFO["actuals_lag"]
last_actual_clock = cutoff_clock - RESOLUTION
first_capacity_use = pd.Timestamp(year=YEARS[0], month=1, day=1) + DATA_INFO["capacity_published_after"]

print("DATA_INFO")
print(
    f"  issue time            : {pd.Timestamp(0) + DATA_INFO['issue_clock']:%H:%M} on "
    f"DAY−{DATA_INFO['issue_days_before'].days}"
)
print(
    f"  actuals lag           : {describe(DATA_INFO['actuals_lag'])} -> availability cutoff "
    f"{pd.Timestamp(0) + cutoff_clock:%H:%M}, last usable actual ROW stamped "
    f"{pd.Timestamp(0) + last_actual_clock:%H:%M} on DAY−{DATA_INFO['issue_days_before'].days}"
)
print(
    f"  capacity publication  : {describe(DATA_INFO['capacity_published_after'])} after the start of "
    f"its year (the {YEARS[0]} value is usable from {first_capacity_use:%Y-%m-%d %H:%M})"
)
print("WINDOWS")
for name, value in WINDOWS.items():
    print(f"  {name:<22}: {describe(value)}")
print(f"TRAIN_TEST_SPLIT_METHOD : {TRAIN_TEST_SPLIT_METHOD}")

In [ ]:
SEED = 42  # every booster and every random draw in this notebook

# One grid and one parameter set per booster, shared by its direct and hybrid entries: a hybrid is
# tuned over exactly the same configurations as its direct variant.
LGBM_PARAMS = {"learning_rate": 0.05, "random_state": SEED, "verbose": -1}
LGBM_GRID = {"num_leaves": [31, 63], "n_estimators": [300, 800]}
XGB_PARAMS = {"learning_rate": 0.05, "random_state": SEED, "tree_method": "hist"}
XGB_GRID = {"max_depth": [4, 6], "n_estimators": [300, 800]}

MODELS = {
    "sarimax_fourier": {
        "enabled": True,
        "family": "sarimax",
        "architecture": None,
        "params": {
            "order": (1, 0, 1),                  # fixed, no order search; d = 0 (see the KPSS check)
            "trend": "c",                        # intercept of the regression
            "fourier": {pd.Timedelta(days=1): 4, pd.Timedelta(days=7): 3},  # period -> order K
            "exog": ["fc_grid_load", "fc_gen_wind_solar", "holiday"],
        },
        "grid": {},
        "label": "SARIMAX + Fourier",
        "color": "#D9A53A",
    },
    "lgbm_direct": {
        "enabled": True,
        "family": "lightgbm",
        "architecture": "direct",
        "params": LGBM_PARAMS,
        "grid": LGBM_GRID,
        "label": "LightGBM direct",
        "color": "#2C6EBA",
    },
    "lgbm_hybrid": {
        "enabled": True,
        "family": "lightgbm",
        "architecture": "hybrid",
        "params": LGBM_PARAMS,
        "grid": LGBM_GRID,
        "label": "LightGBM hybrid",
        "color": "#2F8F5B",
    },
    "xgb_direct": {
        "enabled": False,
        "family": "xgboost",
        "architecture": "direct",
        "params": XGB_PARAMS,
        "grid": XGB_GRID,
        "label": "XGBoost direct",
        "color": "#E95D0F",
    },
    "xgb_hybrid": {
        "enabled": False,
        "family": "xgboost",
        "architecture": "hybrid",
        "params": XGB_PARAMS,
        "grid": XGB_GRID,
        "label": "XGBoost hybrid",
        "color": "#B10F0F",
    },
}

# Rows outside the registry, with fixed colours. SMARD is drawn dashed.
FIXED = {
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A"},
    "seasonal_naive": {"label": "Seasonal naive (DAY−7)", "color": "#9098A2", "lag": pd.Timedelta(days=7)},
}

registry = pd.DataFrame(
    {
        key: {
            "enabled": m["enabled"],
            "family": m["family"],
            "architecture": m["architecture"] or "—",
            "grid configurations": int(np.prod([len(v) for v in m["grid"].values()])),
            "grid": m["grid"] or "—",
            "label": m["label"],
            "color": m["color"],
        }
        for key, m in MODELS.items()
    }
).T
print(f"MODELS: {sum(m['enabled'] for m in MODELS.values())} of {len(MODELS)} enabled, seed {SEED}")
display(registry)
print("fixed parameters")
for key, m in MODELS.items():
    params = {k: ({describe(p): o for p, o in v.items()} if k == "fourier" else v) for k, v in m["params"].items()}
    print(f"  {key:<16}: {params}")
print(f"outside the registry: {', '.join(FIXED)}")

In [ ]:
# Booster feature groups (spec Behaviour 18). PROVISIONAL: the team's feature-engineering work may
# replace them. They feed the direct boosters and the hybrids' stage 2 only. SARIMAX's inputs sit in
# its registry entry, and the hybrids' stage 1 always uses the two SMARD forecasts plus a trend.
FEATURES = {
    "calendar": True,             # local hour, day of week, month, is_weekend, holiday flag
    "smard_forecast": True,       # fc_grid_load, fc_gen_wind_solar for the target hour
    "lags": True,                 # residual_load at the same local hour on DAY−2 and DAY−7; last actual at the cutoff
    "recent_smard_error": True,   # mean err_grid_load and err_renewables over the window ending at the cutoff
    "capacity": True,             # cap_wind_off + cap_wind_on + cap_solar under the publication rule
}

FEATURE_WINDOWS = {
    "same_hour_lags": [pd.Timedelta(days=2), pd.Timedelta(days=7)],
    "recent_smard_error": pd.Timedelta(hours=24),
}

print("FEATURES (provisional)")
for group, enabled in FEATURES.items():
    print(f"  {group:<20}: {'on' if enabled else 'off'}")
print("FEATURE_WINDOWS")
for name, value in FEATURE_WINDOWS.items():
    shown = ", ".join(describe(v) for v in value) if isinstance(value, list) else describe(value)
    print(f"  {name:<20}: {shown}")

In [ ]:
INTERVAL = {"level": 0.95}      # empirical prediction interval, calibrated on the validation year

PLOT_MODEL = None               # None: the registry model with the lowest test MAE; or a key, e.g. "lgbm_hybrid"
PLOT_SPLIT_METHOD = "rolling"   # falls back to "static" when rolling is switched off

EXPORT_ENABLED = False          # True writes data/models/model_*.csv; the folder is not created here

print(f"INTERVAL          : {INTERVAL['level']:.0%} prediction interval")
print(f"PLOT_MODEL        : {PLOT_MODEL if PLOT_MODEL else 'None -> lowest test MAE among registry models'}")
print(f"PLOT_SPLIT_METHOD : {PLOT_SPLIT_METHOD}")
print(f"EXPORT_ENABLED    : {EXPORT_ENABLED}")

### 1.4 SMARD's hourly errors

`data/metrics/smard_forecast_errors_hourly.csv` is written by
[`forecast-metrics-claude.ipynb`](../02_forecast_metrics/forecast-metrics-claude.ipynb) (spec 04). It
is used twice:

- to **re-score SMARD** on this notebook's common hours, instead of recomputing SMARD from
  `data/smard.csv`
- as the source of the `recent_smard_error` feature group (`err_grid_load`, `err_renewables`)

It stays in its own frame, `smard_errors`, and is never merged into `time_series`. The cell stops if
the file is missing, or if its timestamps differ from `time_series.index`: that would be an export
from another snapshot of `data/smard.csv`.

In [ ]:
SMARD_ERRORS = DATA_DIR / "metrics" / "smard_forecast_errors_hourly.csv"
SMARD_ERRORS_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SMARD_ERROR_COLUMNS = ["residual_load", "fc_residual_load", "err_residual_load", "err_grid_load", "err_renewables"]

if not SMARD_ERRORS.exists():
    raise FileNotFoundError(
        f"{SMARD_ERRORS} not found. data/metrics/ is gitignored, so the file is not in a fresh clone — "
        f"regenerate it by running {SMARD_ERRORS_SOURCE} top to bottom."
    )

smard_errors = pd.read_csv(SMARD_ERRORS)
smard_errors["timestamp"] = pd.to_datetime(smard_errors["timestamp"], format="%Y-%m-%d %H:%M:%S")
smard_errors = smard_errors.set_index("timestamp")

missing_columns = sorted(set(SMARD_ERROR_COLUMNS) - set(smard_errors.columns))
if missing_columns:
    raise ValueError(f"{SMARD_ERRORS.name} lacks {missing_columns} — re-run {SMARD_ERRORS_SOURCE}.")

if not smard_errors.index.equals(time_series.index):
    raise ValueError(
        f"{SMARD_ERRORS.name} does not match data/smard.csv: "
        f"{len(smard_errors.index.difference(time_series.index)):,} timestamps only in the errors file, "
        f"{len(time_series.index.difference(smard_errors.index)):,} only in smard.csv "
        f"(errors file {smard_errors.index.min()} -> {smard_errors.index.max()}). It is a stale export "
        f"from another snapshot — re-run {SMARD_ERRORS_SOURCE} top to bottom."
    )

print(f"smard_errors    : {len(smard_errors):,} rows, index identical to time_series")
print(f"range           : {smard_errors.index.min()}  ->  {smard_errors.index.max()}")
print(f"columns used    : {SMARD_ERROR_COLUMNS}")

### 1.5 Initial self-check

Structural checks on the loaded data and on the configuration, free of any hardcoded row count or
date. The closing self-check re-runs the data invariants and compares against `LOADED`.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

# The configuration is internally consistent.
REGISTRY_KEYS = {"enabled", "family", "architecture", "params", "grid", "label", "color"}
for key, m in MODELS.items():
    assert REGISTRY_KEYS <= set(m), f"{key}: missing {REGISTRY_KEYS - set(m)}"
    assert m["family"] in {"sarimax", "lightgbm", "xgboost"}, (key, m["family"])
    if m["family"] == "sarimax":
        assert "fc_residual_load" not in m["params"]["exog"], f"{key}: fc_residual_load is never an input"
    else:
        assert m["architecture"] in {"direct", "hybrid"}, (key, m["architecture"])
    if m["architecture"] == "hybrid":
        direct = [d for d in MODELS.values() if d["family"] == m["family"] and d["architecture"] == "direct"]
        assert direct and m["grid"] == direct[0]["grid"], f"{key}: a hybrid uses its direct variant's grid"
assert not set(MODELS) & set(FIXED), "a registry key collides with a row outside the registry"
assert any(m["enabled"] for m in MODELS.values()), "switch on at least one registry model"
assert set(TRAIN_TEST_SPLIT_METHOD) == {"static", "rolling"}, TRAIN_TEST_SPLIT_METHOD
assert any(TRAIN_TEST_SPLIT_METHOD.values()), "switch on at least one split method"
assert PLOT_MODEL is None or MODELS.get(PLOT_MODEL, {}).get("enabled"), f"PLOT_MODEL {PLOT_MODEL!r} is not an enabled registry key"
assert PLOT_SPLIT_METHOD in TRAIN_TEST_SPLIT_METHOD, PLOT_SPLIT_METHOD
assert set(FEATURES) == {"calendar", "smard_forecast", "lags", "recent_smard_error", "capacity"}, FEATURES
assert 0 < INTERVAL["level"] < 1, INTERVAL
assert isinstance(EXPORT_ENABLED, bool)

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique, columns == SERIES + DERIVED")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")
print(f"  configuration consistent: {sum(m['enabled'] for m in MODELS.values())} registry models, "
      f"split methods {[k for k, v in TRAIN_TEST_SPLIT_METHOD.items() if v]}")

---

## 2 Forecast setting

Every model and every training row obeys one rule for **what is known when**. Defaults from
`DATA_INFO` in brackets:

| Item | Rule |
|---|---|
| `ISSUE_TIME` | the forecast for delivery day `DAY` is issued on `DAY−1` (18:00 local) |
| `AVAILABILITY_CUTOFF` | `ISSUE_TIME − actuals lag` (3 h, so 15:00 on `DAY−1`) |
| actuals | a row stamped `t` (interval start) is usable if `t + resolution ≤ AVAILABILITY_CUTOFF`: up to the row stamped 14:00 |
| SMARD forecasts for `DAY` | available: both components are published by 18:00 on `DAY−1` |
| SMARD forecasts for the rest of `DAY−1` | available: published on `DAY−2` |
| target hours | every local hour of `DAY`: 24, or 23 on the spring DST day (the autumn fold is collapsed by SMARD) |
| training rows | delivery days whose target hours are all observed by the fit's cutoff, each row built as it would have been at its own issue time |

`ISSUE_TIME` and `AVAILABILITY_CUTOFF` are **per-day timestamps** from `forecast_setting`, never
constants. The truncation leakage test that proves the rule holds needs the feature builder and
SARIMAX, so it closes the Models section.

### 2.1 The forecast-setting helper

`forecast_setting(days)` returns, per delivery day, the issue time, the availability cutoff and the
target hours. `SETTING` holds it for every day of the record, `ROW_SETTING` the same times per row.

In [ ]:
# Delivery day of every row: its local calendar date as a local midnight (the DERIVED "date" boundary).
DAY_OF_ROW = time_series.index.normalize()
TARGET_HOURS = time_series.index.groupby(DAY_OF_ROW)   # delivery day -> its local hours in the record
DAYS = pd.DatetimeIndex(sorted(TARGET_HOURS))


def forecast_setting(days):
    """The forecast setting of each delivery day in `days` (local midnights).

    One row per day: `issue_time` (ISSUE_TIME), `cutoff` (AVAILABILITY_CUTOFF) and `target_hours`,
    the day's local hours as they exist in the record. Every model, feature and training row takes
    its times from here.
    """
    days = pd.DatetimeIndex(days)
    issue_time = days - DATA_INFO["issue_days_before"] + DATA_INFO["issue_clock"]
    return pd.DataFrame(
        {
            "issue_time": issue_time,
            "cutoff": issue_time - DATA_INFO["actuals_lag"],
            "target_hours": [TARGET_HOURS[day] for day in days],
        },
        index=days,
    )


def available(stamps, cutoff):
    """True where an observation stamped `stamps` (interval start) has ended by `cutoff`."""
    return stamps + RESOLUTION <= cutoff


SETTING = forecast_setting(DAYS)
ROW_SETTING = pd.DataFrame(
    {
        "day": DAY_OF_ROW,
        "issue_time": SETTING["issue_time"].reindex(DAY_OF_ROW).to_numpy(),
        "cutoff": SETTING["cutoff"].reindex(DAY_OF_ROW).to_numpy(),
    },
    index=time_series.index,
)

hours_per_day = SETTING["target_hours"].map(len)
dst_days = SETTING.index[hours_per_day < EXPECTED_OBS_PER_DAY]


def setting_summary(day):
    """One readable row of the forecast setting for delivery day `day`."""
    row = SETTING.loc[day]
    last_actual = row["cutoff"] - RESOLUTION
    # Counted in rows, not clock hours: the missing spring-DST hour shortens the horizon by one row.
    last_position = time_series.index.searchsorted(last_actual, side="right") - 1
    rows_ahead = time_series.index.get_indexer(row["target_hours"]) - last_position
    return {
        "DAY": f"{day:%Y-%m-%d} ({DAY_NAMES[day.dayofweek]})",
        "ISSUE_TIME": f"{row['issue_time']:%Y-%m-%d %H:%M}",
        "AVAILABILITY_CUTOFF": f"{row['cutoff']:%Y-%m-%d %H:%M}",
        "last usable actual row": f"{last_actual:%Y-%m-%d %H:%M}",
        "target hours": len(row["target_hours"]),
        "rows after last usable actual": f"{rows_ahead.min()}–{rows_ahead.max()}",
    }


print(f"delivery days     : {len(SETTING):,}  ({DAYS[0]:%Y-%m-%d} .. {DAYS[-1]:%Y-%m-%d})")
print(f"target hours/day  : {hours_per_day.value_counts().sort_index(ascending=False).to_dict()}")
print(f"spring DST days   : {len(dst_days)}, each with {hours_per_day[dst_days].min()} target hours")
print("\nForecast setting of two delivery days: the record's latest spring DST day and its last day")
display(pd.DataFrame([setting_summary(dst_days[-1]), setting_summary(DAYS[-1])]).set_index("DAY"))

### 2.2 Feature availability rules

| Input | Usable at `ISSUE_TIME` | Why |
|---|---|---|
| actuals: `residual_load`, and SMARD's errors (they need the actual) | rows with `t + resolution ≤ AVAILABILITY_CUTOFF` | actuals appear on smard.de about 2 h late |
| SMARD component forecasts for the target hour | yes | published by 18:00 on `DAY−1` |
| SMARD component forecasts for `DAY−1` from the cutoff on | yes | published on `DAY−2`; SARIMAX's exogenous inputs |
| calendar: hour, weekday, month, weekend, holiday | always | known in advance |
| installed capacity | the latest value published by `ISSUE_TIME` | publication rule, §2.3 |

The cell below checks the same-hour lags in `FEATURE_WINDOWS` against the rule, for every row of the
record. A lag that reaches past the cutoff (e.g. `DAY−1`, whose afternoon and evening come after
15:00) stops the notebook instead of leaking silently. The two windows that end at the cutoff hold by
definition; the leakage test at the end of the Models section checks every built feature.

In [ ]:
cutoffs = ROW_SETTING["cutoff"].to_numpy()
issue_day = f"DAY−{DATA_INFO['issue_days_before'].days}"
availability = []

for lag in FEATURE_WINDOWS["same_hour_lags"]:
    source = time_series.index - lag
    usable = available(source, cutoffs)
    if not usable.all():
        raise ValueError(
            f"same-hour lag {describe(lag)} reaches past the availability cutoff for "
            f"{(~usable).sum():,} rows: it would leak. Use a lag the forecast setting allows."
        )
    availability.append({
        "feature": f"residual_load at the same local hour, {describe(lag)} earlier",
        "source rows": "one per target hour",
        "smallest margin to the cutoff": describe((cutoffs - (source + RESOLUTION)).min()),
    })

# These two windows are defined to end at the cutoff; the leakage test confirms the built features do.
for feature, window in [
    ("last residual_load before the cutoff", RESOLUTION),
    ("mean err_grid_load and err_renewables", FEATURE_WINDOWS["recent_smard_error"]),
]:
    availability.append({
        "feature": feature,
        "source rows": (
            f"{describe(window)} ending at the cutoff, last row "
            f"{pd.Timestamp(0) + cutoff_clock - RESOLUTION:%H:%M} on {issue_day}"
        ),
        "smallest margin to the cutoff": describe(pd.Timedelta(0)),
    })

print("Actual-derived features against the availability rule (lags checked on every row of the record)")
display(pd.DataFrame(availability).set_index("feature"))

### 2.3 Capacity publication rule

The `cap_*` columns hold one value per calendar year. The project treats year `Y`'s value as
**published on 1 January of `Y`, 00:00** (team decision, `DATA_INFO`). A delivery day uses the
latest value published at or before its issue time:

- Most delivery days use the **current** calendar year's value.
- **1 January uses the previous year's value.** It is issued at 18:00 on 31 December, before the new
  year's value is out. This is why the rule is a publication time, not a fixed year offset.
- Only the record's first day is issued before the first publication and has **no** capacity value.
  It is unforecastable anyway (§2.4), and the default 24-month windows never reach it.

Growth of the fleet within a year is invisible in a yearly step value.

In [ ]:
CAP_COLUMNS = ["cap_wind_off", "cap_wind_on", "cap_solar"]


def capacity_table(frame):
    """One row per calendar year in `frame`: its cap_* values, their total (MW), and when they count as published."""
    per_year = frame[CAP_COLUMNS].groupby(frame.index.year)
    if (per_year.nunique() > 1).any().any():
        raise ValueError("a cap_* column changes within a calendar year: the yearly-step assumption fails")
    table = per_year.first().rename_axis("capacity_year")
    table["cap_total"] = table[CAP_COLUMNS].sum(axis=1)
    table["published_at"] = [
        pd.Timestamp(year=year, month=1, day=1) + DATA_INFO["capacity_published_after"] for year in table.index
    ]
    return table


def published_capacity(issue_times, table):
    """Capacity year and total (MW) of the latest value published at or before each issue time.

    `issue_times` must be sorted. Empty (NaN) before the first publication. Rows keep the input order.
    """
    left = pd.DataFrame({"issue_time": pd.DatetimeIndex(issue_times)})
    right = table.reset_index()[["published_at", "capacity_year", "cap_total"]]
    # merge_asof refuses mixed datetime units, and Timestamp arithmetic can change the unit.
    right["published_at"] = right["published_at"].astype(left["issue_time"].dtype)
    merged = pd.merge_asof(
        left,
        right,
        left_on="issue_time",
        right_on="published_at",
        direction="backward",
    )
    return merged[["capacity_year", "cap_total"]].astype({"capacity_year": "Int64"})


CAPACITY = capacity_table(time_series)
DAY_CAPACITY = published_capacity(SETTING["issue_time"], CAPACITY).set_axis(SETTING.index)

shown = CAPACITY.assign(
    published_at=CAPACITY["published_at"].dt.strftime("%Y-%m-%d %H:%M"),
    used_in_record=CAPACITY["published_at"] <= SETTING["issue_time"].max(),
)
print("Installed capacity per year (MW) and its publication time")
display(shown.style.format("{:,.0f}", subset=CAP_COLUMNS + ["cap_total"]))

# The record's latest 1 January and its two neighbours, derived from the data.
new_year = DAYS[(DAYS.month == 1) & (DAYS.day == 1)][-1]
around = [new_year - pd.Timedelta(days=1), new_year, new_year + pd.Timedelta(days=1)]
example = pd.DataFrame(
    {
        "ISSUE_TIME": SETTING.loc[around, "issue_time"].dt.strftime("%Y-%m-%d %H:%M"),
        "capacity year used": DAY_CAPACITY.loc[around, "capacity_year"],
        "years before DAY's year": pd.Index(around).year - DAY_CAPACITY.loc[around, "capacity_year"],
        "cap_total (MW)": DAY_CAPACITY.loc[around, "cap_total"].map("{:,.0f}".format),
    },
    index=pd.Index([f"{d:%Y-%m-%d}" for d in around], name="DAY"),
)
print(f"\nThe 1 January consequence, around the record's latest new year ({new_year:%Y-%m-%d})")
display(example)

no_capacity = SETTING.index[DAY_CAPACITY["capacity_year"].isna()]
print(f"\ndelivery days without a published capacity value: {len(no_capacity):,}", end="")
print(f" ({no_capacity[0]:%Y-%m-%d} .. {no_capacity[-1]:%Y-%m-%d})" if len(no_capacity) else "")

### 2.4 Unforecastable days

`DAY` cannot be forecast, by **any** model, as soon as a SMARD component forecast (`fc_grid_load`,
`fc_gen_wind_solar`) is missing for an hour of `DAY`, or for an hour of `DAY−1` from the cutoff on:
SARIMAX takes those hours as exogenous inputs. Such a day has no input rows. It is a **data
exclusion**, not a model failure: it is left out of training and scoring and counted here, from the
data. The record's first day is excluded for the same reason: its `DAY−1` lies outside the record.

In [ ]:
FORECAST_INPUTS = ["fc_grid_load", "fc_gen_wind_solar"]

# Cumulative count of rows lacking a component forecast: any [start, end) row range is then one subtraction.
missing_input = time_series[FORECAST_INPUTS].isna().any(axis=1).to_numpy()
cum_missing = np.concatenate([[0], np.cumsum(missing_input)])

first_unavailable = time_series.index.searchsorted(SETTING["cutoff"])   # first DAY−1 row after the cutoff
day_start = time_series.index.searchsorted(SETTING.index)
day_end = time_series.index.searchsorted(SETTING.index + pd.Timedelta(days=1))

EXCLUSION = pd.DataFrame(
    {
        "forecast missing on DAY": cum_missing[day_end] - cum_missing[day_start] > 0,
        "forecast missing on DAY−1 after the cutoff": cum_missing[day_start] - cum_missing[first_unavailable] > 0,
        "DAY−1 outside the record": SETTING["cutoff"] < time_series.index[0],
    },
    index=SETTING.index,
)
FORECASTABLE = ~EXCLUSION.any(axis=1)

excluded = EXCLUSION[~FORECASTABLE]
print(f"forecastable delivery days : {FORECASTABLE.sum():,} of {len(FORECASTABLE):,}")
print(f"unforecastable             : {len(excluded)}")
display(
    excluded.apply(lambda row: ", ".join(row.index[row]), axis=1)
    .rename("reason")
    .set_axis(pd.Index([f"{d:%Y-%m-%d}" for d in excluded.index], name="DAY"))
    .to_frame()
)

### 2.5 Self-check

In [ ]:
assert SETTING.index.equals(DAYS) and DAYS.is_unique and DAYS.is_monotonic_increasing
assert (SETTING["issue_time"] - SETTING["cutoff"] == DATA_INFO["actuals_lag"]).all()
assert (SETTING["issue_time"] < SETTING.index).all(), "a forecast is issued after its delivery day starts"
assert sum(map(len, SETTING["target_hours"])) == len(time_series), "target hours do not partition the record"
assert all((hours.normalize() == day).all() for day, hours in SETTING["target_hours"].items())
assert ROW_SETTING.index.equals(time_series.index) and ROW_SETTING.notna().all().all()

# Capacity: never a value published after the issue time, never from a later calendar year.
used = DAY_CAPACITY["capacity_year"].notna().to_numpy()
published_at = CAPACITY.loc[DAY_CAPACITY["capacity_year"][used], "published_at"].to_numpy()
assert (published_at <= SETTING["issue_time"].to_numpy()[used]).all(), "capacity used before its publication"
assert (DAY_CAPACITY["capacity_year"][used].to_numpy() <= SETTING.index.year[used]).all()

assert EXCLUSION.index.equals(SETTING.index) and FORECASTABLE.dtype == bool
assert list(time_series.columns) == SERIES + DERIVED, "a section 2 cell persisted a column onto time_series"

print("section 2 self-check passed")
print(f"  {len(SETTING):,} delivery days, target hours partition the record, cutoff = issue time − {describe(DATA_INFO['actuals_lag'])}")
print(f"  capacity never used before its publication; {(~FORECASTABLE).sum()} unforecastable days")

---

## 3 Windows and split methods

- **Test window:** the last 365 **complete** delivery days, ending with the last day whose hours all
  carry the actual and every SMARD forecast. **Validation window:** the 365 delivery days before it.
  Both are derived from the data.
- **Training rows of a fit:** the delivery days inside the 24-month window ending at the fit's
  cutoff whose target hours are all observed by that cutoff (up to `DAY−2` for a fit issued on
  `DAY−1`). Unforecastable days and days with a missing actual are left out.
- **Tuning:** a rolling walk-forward over the validation year, once per grid configuration. The
  configuration with the lowest validation MAE is selected and **frozen**: both split methods reuse
  it unchanged, and refits never re-run the search.
- **Static:** one fit issued for the first test day. It forecasts the whole test year with daily
  updated inputs and frozen parameters.
- **Rolling:** the same first fit, then a refit every `refit_every` on the sliding 24-month window.
- Test-year fits train on windows that **include the validation year**: tuning only chose the
  configuration.

Both split methods forecast **identical test hours** with equal window lengths, so the only
difference between them is the refitting.

### 3.1 Test and validation windows

In [ ]:
JOINT_COLUMNS = ["residual_load", "fc_residual_load", "fc_grid_load", "fc_gen_wind_solar"]

by_day = pd.DataFrame(
    {
        "actual": time_series["residual_load"].notna(),
        "joint": time_series[JOINT_COLUMNS].notna().all(axis=1),
        "stamp": time_series.index,
    },
    index=time_series.index,
).groupby(DAY_OF_ROW)
DAY_INFO = pd.DataFrame(
    {
        "rows": by_day.size(),
        "first_row": by_day["stamp"].min(),
        "last_row": by_day["stamp"].max(),
        "actual_complete": by_day["actual"].all(),
        "jointly_observed": by_day["joint"].all(),
    }
)
assert DAY_INFO.index.equals(DAYS)

# A day covers its full clock span when it starts at 00:00, ends at the last hour before midnight and
# meets the completeness rule (the spring-DST day has 23 rows and passes).
full_span = (
    (DAY_INFO["first_row"] == DAY_INFO.index)
    & (DAY_INFO["last_row"] == DAY_INFO.index + pd.Timedelta(days=1) - RESOLUTION)
    & (DAY_INFO["rows"] >= MIN_OBS_PER_DAY)
)
TRAINABLE = FORECASTABLE & full_span & DAY_INFO["actual_complete"]
COMPLETE = full_span & DAY_INFO["jointly_observed"]

TEST_END = COMPLETE.index[COMPLETE][-1]
TEST_DAYS = DAYS[(DAYS > TEST_END - WINDOWS["test"]) & (DAYS <= TEST_END)]
VAL_DAYS = DAYS[(DAYS >= TEST_DAYS[0] - WINDOWS["validation"]) & (DAYS < TEST_DAYS[0])]


def hours_of(days):
    """The record's local hours of the delivery days in `days`."""
    return time_series.index[DAY_OF_ROW.isin(days)]


# Spec 04's trailing_365 window: 365 days of hours ending at the last jointly observed hour.
jointly = time_series[JOINT_COLUMNS].notna().all(axis=1)
JOINT_END = time_series.index[jointly][-1]
trailing_365 = time_series.index[(time_series.index > JOINT_END - WINDOWS["test"]) & (time_series.index <= JOINT_END)]

windows = pd.DataFrame(
    {
        name: {
            "first day": f"{days[0]:%Y-%m-%d}",
            "last day": f"{days[-1]:%Y-%m-%d}",
            "days": len(days),
            "hours": len(hours_of(days)),
            "unforecastable days": int((~FORECASTABLE.loc[days]).sum()),
            "days without a complete actual": int((~TRAINABLE.loc[days] & FORECASTABLE.loc[days]).sum()),
        }
        for name, days in [("validation", VAL_DAYS), ("test", TEST_DAYS)]
    }
).T
print("Test and validation windows (delivery days, local time)")
display(windows)

print(f"last jointly observed hour : {JOINT_END:%Y-%m-%d %H:%M}")
if hours_of(TEST_DAYS).equals(trailing_365):
    print(f"test window                : identical to spec 04's trailing_365 window ({len(trailing_365):,} h)")
else:
    print(
        f"test window                : ends {TEST_END:%Y-%m-%d}; the incomplete day after it is left out, "
        f"so it differs from spec 04's trailing_365 window ({len(trailing_365):,} h)"
    )

### 3.2 Fit schedule and training windows

One schedule per run. The two validation runs exist for tuning and for calibrating the prediction
bands (§5): the rolling band from the tuning walk-forward, the static band from one frozen fit over
the validation year.

In [ ]:
def fit_days_for(window_days, refit_every):
    """Issue days of the fits serving `window_days`: the first day, then one every `refit_every` (None: never)."""
    if refit_every is None:
        return window_days[:1]
    return pd.date_range(window_days[0], window_days[-1], freq=refit_every)


def training_days(fit_day):
    """Delivery days a fit issued for `fit_day` trains on (see §3's training-row rule)."""
    cutoff = SETTING.at[fit_day, "cutoff"]
    in_window = DAYS >= cutoff - WINDOWS["train"]
    observed_by_cutoff = available(DAY_INFO["last_row"], cutoff).to_numpy()
    return DAYS[in_window & observed_by_cutoff & TRAINABLE.to_numpy()]


RUNS = {
    "validation_rolling": {
        "days": VAL_DAYS, "refit_every": WINDOWS["refit_every"], "enabled": True,
        "purpose": "tuning; calibrates the rolling band",
    },
    "validation_static": {
        "days": VAL_DAYS, "refit_every": None, "enabled": TRAIN_TEST_SPLIT_METHOD["static"],
        "purpose": "calibrates the static band",
    },
    "static": {
        "days": TEST_DAYS, "refit_every": None, "enabled": TRAIN_TEST_SPLIT_METHOD["static"],
        "purpose": "test year, one frozen fit",
    },
    "rolling": {
        "days": TEST_DAYS, "refit_every": WINDOWS["refit_every"], "enabled": TRAIN_TEST_SPLIT_METHOD["rolling"],
        "purpose": f"test year, refit every {describe(WINDOWS['refit_every'])}",
    },
}
for run in RUNS.values():
    run["fit_days"] = fit_days_for(run["days"], run["refit_every"])

for name, run in RUNS.items():
    window_start = SETTING.at[run["fit_days"][0], "cutoff"] - WINDOWS["train"]
    if window_start < time_series.index[0]:
        raise ValueError(
            f"{name}: the first fit's {describe(WINDOWS['train'])} training window starts {window_start:%Y-%m-%d}, "
            f"before the record ({time_series.index[0]:%Y-%m-%d}). Shorten WINDOWS['train'] or re-fetch more history."
        )


def span(days):
    return f"{days[0]:%Y-%m-%d} .. {days[-1]:%Y-%m-%d} ({len(days)} days)"


schedule = pd.DataFrame(
    {
        name: {
            "purpose": run["purpose"],
            "enabled": run["enabled"],
            "fits": len(run["fit_days"]),
            "forecast days": span(run["days"]),
            "first fit: training days": span(training_days(run["fit_days"][0])),
            "last fit: training days": span(training_days(run["fit_days"][-1])),
        }
        for name, run in RUNS.items()
    }
).T
print("Fit schedule per run")
display(schedule)

if RUNS["static"]["enabled"] and RUNS["rolling"]["enabled"]:
    shared_until = RUNS["rolling"]["fit_days"][1] - pd.Timedelta(days=1) if len(RUNS["rolling"]["fit_days"]) > 1 else TEST_DAYS[-1]
    print(
        f"static and rolling share their first fit, so they forecast identically for "
        f"{TEST_DAYS[0]:%Y-%m-%d} .. {shared_until:%Y-%m-%d}: that stretch cannot differ between them."
    )

### 3.3 The split engine

Each model family supplies one function, registered in `FAMILY_FIT` in the Models section:
`fit(key, config, train_days, forecast_days)` returns `predict(day)`, which forecasts every target
hour of one delivery day. The engine calls it per scheduled fit and applies the failure rules:

- A fit that **raises** leaves its days empty until the next successful fit. The previous fit does
  not keep forecasting, and no other model fills in.
- A day whose forecast **raises** stays empty.
- A **convergence warning** is not a failure: the forecast is kept and the warning is counted.

`tune` runs the validation walk-forward once per grid configuration and selects the lowest MAE on
the hours all configurations forecast. `evaluate_model` then freezes that configuration and runs
every enabled split method.

In [ ]:
FAMILY_FIT = {}   # family -> fit function, filled in the Models section


def grid_configurations(model):
    """Every configuration of a registry entry's tuning grid, merged over its fixed parameters."""
    names = list(model["grid"])
    return [{**model["params"], **dict(zip(names, values))} for values in itertools.product(*model["grid"].values())]


def run_split(key, config, run):
    """Forecast the days of `run` with model `key` under `config`, each day by the latest fit at or before it.

    Returns the hourly forecast over the run's hours (empty where no forecast exists) and one log row per fit.
    """
    days, fit_days = RUNS[run]["days"], RUNS[run]["fit_days"]
    serving_fit = fit_days.searchsorted(days, side="right") - 1
    pieces, log = [], []

    for i, fit_day in enumerate(fit_days):
        serve = days[(serving_fit == i) & FORECASTABLE.loc[days].to_numpy()]
        train = training_days(fit_day)
        entry = {"fit_day": fit_day, "train_days": len(train), "forecast_days": len(serve), "status": "ok",
                 "failed_days": 0, "convergence_warnings": 0, "other_warnings": 0}

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            start = time.perf_counter()
            try:
                predict = FAMILY_FIT[MODELS[key]["family"]](key, config, train, serve)
            except Exception as error:
                predict, entry["status"] = None, f"failed: {type(error).__name__}: {error}"
            entry["fit_seconds"] = time.perf_counter() - start

            start = time.perf_counter()
            for day in serve if predict is not None else []:
                try:
                    forecast = predict(day)
                except Exception:
                    entry["failed_days"] += 1
                    continue
                assert forecast.index.equals(SETTING.at[day, "target_hours"]), f"{key}: {day:%Y-%m-%d} is not forecast hour by hour"
                pieces.append(forecast)
            entry["forecast_seconds"] = time.perf_counter() - start

        entry["convergence_warnings"] = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
        entry["other_warnings"] = len(caught) - entry["convergence_warnings"]
        log.append(entry)

    forecast = pd.concat(pieces) if pieces else pd.Series(dtype=float)
    return forecast.reindex(hours_of(days)).astype(float), pd.DataFrame(log)


def tune(key):
    """Validation walk-forward for every grid configuration of `key`; selects the lowest MAE."""
    actual = time_series["residual_load"]
    trials = []
    for config in grid_configurations(MODELS[key]):
        forecast, log = run_split(key, config, "validation_rolling")
        # A configuration without a single forecast is a failure; it must not empty the others' hours.
        trials.append({"config": config, "forecast": forecast, "log": log, "usable": forecast.notna().any()})

    hours = actual.index[actual.notna()]
    for t in trials:
        if t["usable"]:
            hours = hours.intersection(t["forecast"].index[t["forecast"].notna()])

    rows = []
    for t in trials:
        grid_point = {name: t["config"][name] for name in MODELS[key]["grid"]}
        mae = (t["forecast"][hours] - actual[hours]).abs().mean() if t["usable"] else np.nan
        rows.append({**grid_point, "MAE": mae, "hour_count": len(hours) if t["usable"] else 0,
                     "failed fits": int((t["log"]["status"] != "ok").sum()),
                     "seconds": t["log"][["fit_seconds", "forecast_seconds"]].to_numpy().sum()})
    table = pd.DataFrame(rows)
    best = table["MAE"].idxmin() if table["MAE"].notna().any() else None
    return {
        "table": table,
        "selected": trials[best]["config"] if best is not None else None,
        "forecast": trials[best]["forecast"] if best is not None else None,
        "log": trials[best]["log"] if best is not None else None,
        "seconds": table["seconds"].sum(),
    }


def evaluate_model(key):
    """Tune `key` on the validation year, then run every enabled split method with the frozen configuration."""
    tuning = tune(key)
    result = {"tuning": tuning, "runs": {}}
    if tuning["selected"] is None:
        return result
    result["runs"]["validation_rolling"] = (tuning["forecast"], tuning["log"])
    for run in ["validation_static", "static", "rolling"]:
        if RUNS[run]["enabled"]:
            result["runs"][run] = run_split(key, tuning["selected"], run)
    return result

### 3.4 Self-check

In [ ]:
assert len(TEST_DAYS) == WINDOWS["test"].days and len(VAL_DAYS) == WINDOWS["validation"].days
assert VAL_DAYS[-1] + pd.Timedelta(days=1) == TEST_DAYS[0], "validation must end the day before the test window"
assert not VAL_DAYS.intersection(TEST_DAYS).size, "validation and test windows overlap"
assert COMPLETE.loc[TEST_END] and not COMPLETE.loc[DAYS > TEST_END].any(), "the test window must end on the last complete day"

for name, run in RUNS.items():
    for fit_day in run["fit_days"]:
        train = training_days(fit_day)
        assert available(DAY_INFO.loc[train, "last_row"], SETTING.at[fit_day, "cutoff"]).all()
        assert TRAINABLE.loc[train].all() and (train < fit_day).all()
        if name.startswith("validation"):
            assert (train < TEST_DAYS[0]).all(), f"{name}: tuning must never see a test day"

assert RUNS["static"]["fit_days"][0] == RUNS["rolling"]["fit_days"][0], "static and rolling share the first fit"
assert training_days(RUNS["static"]["fit_days"][0]).intersection(VAL_DAYS).size, "test fits must include the validation year"
assert list(time_series.columns) == SERIES + DERIVED, "a section 3 cell persisted a column onto time_series"

print("section 3 self-check passed")
print(f"  validation {VAL_DAYS[0]:%Y-%m-%d} .. {VAL_DAYS[-1]:%Y-%m-%d}, test {TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d}, no overlap")
print("  every fit trains only on days observed by its cutoff; no validation run trains on a test day")